In [12]:
# 1) Imports and configuration
import os
import time
import random
import numpy as np
import torch
from torch.utils.data import IterableDataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

data_path = "/home/pk222/projects/PDEControl_DPC/datasets/heat_smooth_f_dataset.npz"
samples_to_load = 100
seed = 32
train_frac = 0.8
batch_size = 75
num_steps = 1000
log_iter = 10
lr = 1e-3
transition_steps = 2000
decay_rate = 0.9
sample_id = 5
num_samples_to_evaluate = 10

resume_path = ""
resume_strict = True
save_best = True
save_last = True

# RNO preprocessing hyperparameters
sub = 1
T = 16
n_intervals = 5
train_up_to_tipping_point = False
tipping_data_split_prop = 0.67
debug_preprocessing_shapes = True


In [13]:
# 2) Reproducibility and device
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required but not available.")
device = torch.device("cuda")
print(f"Using device: {device}")


Using device: cuda


In [14]:
# 3) Data loading and RNO preprocessing
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found: {data_path}")

dataset = np.load(data_path)
solutions = torch.from_numpy(dataset["solutions"][:samples_to_load]).float()
controls = torch.from_numpy(dataset["controls"][:samples_to_load]).float()
x_cord = torch.from_numpy(dataset["x"]).reshape(-1, 1).float()
dt = float(dataset["dt"])

print(f"Loaded {samples_to_load}/{dataset['solutions'].shape[0]} samples:")
print(f"solutions: {tuple(solutions.shape)}")
print(f"controls: {tuple(controls.shape)}")
print(f"x: {tuple(x_cord.shape)}")
print(f"dt: {dt}")

Loaded 100/3000 samples:
solutions: (100, 401, 100)
controls: (100, 400, 4)
x: (100, 1)
dt: 0.001


In [15]:
def round_down(num, divisor):
    return num - (num % divisor)


def chunk_time_axis(arr, chunk_size):
    # arr shape: (ntraj, nt, nfeat) -> (ntraj, n_chunks, chunk_size, nfeat)
    chunks = torch.split(arr, chunk_size, dim=1)
    return torch.stack(chunks, dim=1)

In [16]:
# 1) Align solution/control time lengths and crop to multiple of T
ntraj = min(solutions.shape[0], controls.shape[0])
nt_shared = min(solutions.shape[1] - 1, controls.shape[1])
nt_usable = round_down(nt_shared, T)

solutions = solutions[:ntraj, :nt_usable, :]
controls = controls[:ntraj, :nt_usable, :]

# 2) Optional spatial subsampling on solution/grid only
solutions = solutions[:, :, ::sub]
x_cord = x_cord[::sub]

# 3) Optional pre-tipping crop
if train_up_to_tipping_point:
    tipping_idx = round_down(int(tipping_data_split_prop * solutions.shape[1]), T)
    solutions = solutions[:, :tipping_idx, :]
    controls = controls[:, :tipping_idx, :]

ntraj, nt, nx = solutions.shape
nf = controls.shape[-1]
n_chunks = nt // T

print(f"After preprocessing:")
print(f"solutions: {tuple(solutions.shape)}")
print(f"controls: {tuple(controls.shape)}")
print(f"x: {tuple(x_cord.shape)}")
print(f"ntraj: {ntraj}, nt: {nt}, nx: {nx}, nf: {nf}, n_chunks: {n_chunks}")


After preprocessing:
solutions: (100, 400, 100)
controls: (100, 400, 4)
x: (100, 1)
ntraj: 100, nt: 400, nx: 100, nf: 4, n_chunks: 25


Train/test split by trajectory (same strategy as ks/RNO_KS.py)

In [17]:
# 4) Train/test split by trajectory (same strategy as ks/RNO_KS.py)
n_train_traj = int(train_frac * ntraj)
n_test_traj = ntraj - n_train_traj

sol_train = solutions[:n_train_traj]
sol_test = solutions[n_train_traj:]
ctrl_train = controls[:n_train_traj]
ctrl_test = controls[n_train_traj:]

Chunk with T=16: (400 timesteps)

U_chunks: (B, 25, 16, 100, 1)
F_chunks: (B, 25, 16, 4)

In [18]:
# 5) Chunk both streams
sol_train_chunks = chunk_time_axis(sol_train, T).unsqueeze(
    -1
)  # (n_tr, n_chunks, T, nx, 1)
sol_test_chunks = chunk_time_axis(sol_test, T).unsqueeze(-1)
ctrl_train_chunks = chunk_time_axis(ctrl_train, T)  # (n_tr, n_chunks, T, nf)
ctrl_test_chunks = chunk_time_axis(ctrl_test, T)

# Broadcast controls over space so they can be concatenated as channels for RNO
ctrl_train_chunks = ctrl_train_chunks.unsqueeze(3).expand(-1, -1, -1, nx, -1)
ctrl_test_chunks = ctrl_test_chunks.unsqueeze(3).expand(-1, -1, -1, nx, -1)

print(f"After chunking:")
print(f"sol_train_chunks: {tuple(sol_train_chunks.shape)}")
print(f"ctrl_train_chunks: {tuple(ctrl_train_chunks.shape)}")
print(f"sol_test_chunks: {tuple(sol_test_chunks.shape)}")
print(f"ctrl_test_chunks: {tuple(ctrl_test_chunks.shape)}")

After chunking:
sol_train_chunks: (80, 25, 16, 100, 1)
ctrl_train_chunks: (80, 25, 16, 100, 4)
sol_test_chunks: (20, 25, 16, 100, 1)
ctrl_test_chunks: (20, 25, 16, 100, 4)


In [19]:
def build_windows(sol_chunks, ctrl_chunks, n_intervals_inner):
    x_sol, x_ctrl, y_sol = [], [], []
    for traj_sol, traj_ctrl in zip(sol_chunks, ctrl_chunks):
        for i in range(sol_chunks.shape[1] - n_intervals_inner - 1):
            x_sol.append(traj_sol[i : i + n_intervals_inner])
            x_ctrl.append(traj_ctrl[i : i + n_intervals_inner])
            y_sol.append(traj_sol[i + n_intervals_inner])

    x_sol = torch.stack(x_sol, dim=0)
    x_ctrl = torch.stack(x_ctrl, dim=0)
    y_sol = torch.stack(y_sol, dim=0)

    # Concatenate solution + control channels
    x_full = torch.cat([x_sol, x_ctrl], dim=-1)  # (N, n_intervals, T, nx, 1+nf)
    return x_full, y_sol


x_train, y_train = build_windows(sol_train_chunks, ctrl_train_chunks, n_intervals)
x_test, y_test = build_windows(sol_test_chunks, ctrl_test_chunks, n_intervals)

# RNO input format: (batch, timesteps, in_channels, *spatial_dims)
x_train_rno = x_train.permute(0, 1, 4, 2, 3).contiguous()
x_test_rno = x_test.permute(0, 1, 4, 2, 3).contiguous()

# Optional target format aligned with model output before post-permute
y_train_rno = y_train.permute(0, 3, 1, 2).contiguous()
y_test_rno = y_test.permute(0, 3, 1, 2).contiguous()

if debug_preprocessing_shapes:
    print("=== RNO preprocessing debug ===")
    print(f"ntraj={ntraj}, nt={nt}, nx={nx}, nf={nf}, n_chunks={n_chunks}")
    print(f"n_train_traj={n_train_traj}, n_test_traj={n_test_traj}")
    print(f"sol_train_chunks: {tuple(sol_train_chunks.shape)}")
    print(f"ctrl_train_chunks: {tuple(ctrl_train_chunks.shape)}")
    print(f"x_train: {tuple(x_train.shape)}")
    print(f"y_train: {tuple(y_train.shape)}")
    print(f"x_test: {tuple(x_test.shape)}")
    print(f"y_test: {tuple(y_test.shape)}")
    print(f"x_train_rno: {tuple(x_train_rno.shape)}")
    print(f"y_train_rno: {tuple(y_train_rno.shape)}")
    print("=== End preprocessing debug ===")


=== RNO preprocessing debug ===
ntraj=100, nt=400, nx=100, nf=4, n_chunks=25
n_train_traj=80, n_test_traj=20
sol_train_chunks: (80, 25, 16, 100, 1)
ctrl_train_chunks: (80, 25, 16, 100, 4)
x_train: (1520, 5, 16, 100, 5)
y_train: (1520, 16, 100, 1)
x_test: (380, 5, 16, 100, 5)
y_test: (380, 16, 100, 1)
x_train_rno: (1520, 5, 5, 16, 100)
y_train_rno: (1520, 1, 16, 100)
=== End preprocessing debug ===


In [20]:
# Dataloaders for RNO training
train_loader = DataLoader(
    torch.utils.data.TensorDataset(x_train, y_train),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)
test_loader = DataLoader(
    torch.utils.data.TensorDataset(x_test, y_test),
    batch_size=batch_size,
    shuffle=False,
    drop_last=True,
)

xb, yb = next(iter(train_loader))
print(f"train batch x: {tuple(xb.shape)}, y: {tuple(yb.shape)}")
print(f"train batch x_rno: {tuple(xb.permute(0, 1, 4, 2, 3).shape)}")


train batch x: (75, 5, 16, 100, 5), y: (75, 16, 100, 1)
train batch x_rno: (75, 5, 5, 16, 100)


In [21]:
# 4) RNO utilities
import operator
from functools import reduce
from tqdm.auto import tqdm


class LpLoss(object):
    def __init__(self, d=2, p=2, size_average=True, reduction=True):
        assert d > 0 and p > 0
        self.d = d
        self.p = p
        self.reduction = reduction
        self.size_average = size_average

    def rel(self, x, y):
        num_examples = x.size()[0]
        diff_norms = torch.norm(
            x.reshape(num_examples, -1) - y.reshape(num_examples, -1), self.p, 1
        )
        y_norms = torch.norm(y.reshape(num_examples, -1), self.p, 1)
        if self.reduction:
            if self.size_average:
                return torch.mean(diff_norms / y_norms)
            return torch.sum(diff_norms / y_norms)
        return diff_norms / y_norms

    def __call__(self, x, y):
        return self.rel(x, y)


def count_params(model):
    c = 0
    for p in list(model.parameters()):
        c += reduce(operator.mul, list(p.size()), 1)
    return c


In [22]:
# 5) RNO model, optimizer, scheduler, loss
from neuralop.models import RNO

modes1 = 20
modes2 = 20
width = 28
n_layers = 3
domain_padding = [0.1, 0]

# Use data-derived channel dims (expected: in=5, out=1)
in_channels = int(x_train.shape[-1])
out_channels = int(y_train.shape[-1])
print(f"in_channels={in_channels}, out_channels={out_channels}")
if in_channels != 5:
    print("Warning: in_channels is not 5; check preprocessing.")

learning_rate = lr
weight_decay = 1e-4
scheduler_step = 50
scheduler_gamma = 0.5
epochs = 25
save_weights = True

model = RNO(
    n_modes=(modes1, modes2),
    hidden_channels=width,
    in_channels=in_channels,
    out_channels=out_channels,
    n_layers=n_layers,
    domain_padding=domain_padding,
).to(device)

print("Model parameters:", count_params(model))
optimizer = torch.optim.Adam(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=scheduler_step, gamma=scheduler_gamma
)
lploss = LpLoss(size_average=False)


in_channels=5, out_channels=1
Model parameters: 3138365


In [23]:
# 6) RNO training loop
num_train_samples = x_train.shape[0]
num_test_samples = x_test.shape[0]
training_loss_history = []
test_loss_history = []

print("Begin RNO training:")
for ep in range(1, epochs + 1):
    model.train()
    t_start = time.time()
    train_l2 = 0.0

    for x_batch, y_batch in tqdm(train_loader, leave=False):
        x_batch = x_batch.to(device).float()  # (B, n_intervals, T, nx, in_channels)
        y_batch = y_batch.to(device).float()  # (B, T, nx, 1)

        x_in = x_batch.permute(0, 1, 4, 2, 3)
        out = model.predict(x_in, num_steps=1)[:, -1]  # (B, 1, T, nx)
        out = out.permute(0, 2, 3, 1)  # (B, T, nx, 1)

        loss = lploss(out, y_batch)
        train_l2 += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device).float()
            y_batch = y_batch.to(device).float()

            x_in = x_batch.permute(0, 1, 4, 2, 3)
            out = model.predict(x_in, num_steps=1)[:, -1]
            out = out.permute(0, 2, 3, 1)
            test_l2 += lploss(out, y_batch).item()

    train_l2 /= num_train_samples
    test_l2 /= num_test_samples
    training_loss_history.append(train_l2)
    test_loss_history.append(test_l2)

    print(
        f"Epoch {ep:03d}/{epochs} | time={time.time() - t_start:.2f}s | train_l2={train_l2:.4e} | test_l2={test_l2:.4e}"
    )


Begin RNO training:


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 001/25 | time=7.56s | train_l2=3.1141e+00 | test_l2=1.6540e+00


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 002/25 | time=5.43s | train_l2=1.1900e+00 | test_l2=1.1621e+00


...

Epoch 023/25 | time=5.79s | train_l2=8.8677e-01 | test_l2=1.0252e+00


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 024/25 | time=6.35s | train_l2=8.8671e-01 | test_l2=1.0254e+00


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 025/25 | time=5.53s | train_l2=8.8696e-01 | test_l2=1.0253e+00


In [24]:
# 7) Save model weights
if save_weights:
    result_dir = os.path.join(os.getcwd(), f"result_rno_{seed}")
    os.makedirs(result_dir, exist_ok=True)
    model_path = os.path.join(result_dir, "rno_model_last.pt")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epochs": epochs,
            "seed": seed,
            "config": {
                "modes1": modes1,
                "modes2": modes2,
                "width": width,
                "in_channels": in_channels,
                "out_channels": out_channels,
                "n_layers": n_layers,
                "domain_padding": domain_padding,
                "learning_rate": learning_rate,
                "weight_decay": weight_decay,
                "scheduler_step": scheduler_step,
                "scheduler_gamma": scheduler_gamma,
                "T": T,
                "n_intervals": n_intervals,
                "sub": sub,
            },
            "train_loss_history": training_loss_history,
            "test_loss_history": test_loss_history,
        },
        model_path,
    )
    print(f"Saved weights to: {model_path}")
else:
    print("save_weights=False, skipping model save.")


Saved weights to: /home/pk222/projects/tipping-point-forecast/heat/result_rno_32/rno_model_last.pt
